In [ ]:
import torch
print("device_count:", torch.cuda.device_count())
print("is_available:", torch.cuda.is_available())

In [ ]:
import sys, site

user_site = site.getusersitepackages()
if user_site not in sys.path:
    sys.path.insert(0, user_site)

from rapidfireai import Experiment
from rapidfireai.automl import List, RFGridSearch, RFModelConfig, RFLoraConfig, RFSFTConfig
from datasets import Dataset
import json

print("RFLoraConfig:", RFLoraConfig)

In [ ]:
import json
from datasets import Dataset

with open("train.json", "r") as f:
    train_dataset = Dataset.from_list(json.load(f))

with open("validation.json", "r") as f:
    validation_dataset = Dataset.from_list(json.load(f))

print(f"Train: {len(train_dataset)} examples, Validation: {len(validation_dataset)} examples")

In [ ]:
def basic_formatting_function(row):
    import json

    clean_id = row["db_id"].replace(" ", "_").replace("/", "_")
    filepath = f"./schemas/{clean_id}.json"
    with open(filepath) as f:
        schema_data = json.load(f)

    schema = {}
    for table_name in schema_data["table_names_original"]:
        schema[table_name] = []
    for i, name in schema_data["column_names_original"]:
        if i == -1:
            continue
        table_name = schema_data["table_names_original"][i]
        schema[table_name].append(name)

    system_prompt = (
        "You are a schema-linking assistant. "
        "Given a question and a database schema, return ONLY a valid JSON object "
        "that maps table names to relevant column-name lists."
    )
    prompt = (
        f"Database schema: {schema}\n\n"
        f"Question: {row['question']}\n\n"
        "Return a JSON object with only the relevant tables as keys and lists of relevant column names as values. "
        "You MUST include specific column names — do not return empty lists unless a table has no relevant columns. "
        "Example: {\"Orders\": [\"order_id\", \"total\"], \"Customers\": [\"name\"]}"
    )
    answer = json.dumps(row["schema_links"], ensure_ascii=False)
    return {
        "text": (
            f"<|im_start|>system\n{system_prompt}<|im_end|>\n"
            f"<|im_start|>user\n{prompt}<|im_end|>\n"
            f"<|im_start|>assistant\n{answer}<|im_end|>"
        )
    }


def pkfk_formatting_function(row):
    import json

    clean_id = row["db_id"].replace(" ", "_").replace("/", "_")
    filepath = f"./schemas/{clean_id}.json"
    with open(filepath) as f:
        schema_data = json.load(f)

    column_info = schema_data["column_names_original"]
    primary_keys = schema_data.get("primary_keys", [])
    foreign_keys = schema_data.get("foreign_keys", [])
    col_annotations = {}

    for pk in primary_keys:
        if isinstance(pk, list):
            for col_idx in pk:
                col_annotations[col_idx] = "(PK)"
        else:
            col_annotations[pk] = "(PK)"

    for from_idx, to_idx in foreign_keys:
        to_table_idx = column_info[to_idx][0]
        to_table_name = schema_data["table_names_original"][to_table_idx]
        if from_idx in col_annotations and col_annotations[from_idx] == "(PK)":
            col_annotations[from_idx] = f"(PK,FK\u2192{to_table_name})"
        else:
            col_annotations[from_idx] = f"(FK\u2192{to_table_name})"

    schema = {}
    for col_idx, (table_idx, col_name) in enumerate(column_info):
        if table_idx == -1:
            continue
        table_name = schema_data["table_names_original"][table_idx]
        if table_name not in schema:
            schema[table_name] = []
        annotation = col_annotations.get(col_idx, "")
        annotated_col = f"{col_name} {annotation}" if annotation else col_name
        schema[table_name].append(annotated_col)

    system_prompt = (
        "You are a schema-linking assistant. "
        "Given a question and a database schema with PK/FK annotations, return ONLY a valid JSON object "
        "that maps table names to relevant column-name lists (without annotations in the output)."
    )
    prompt = (
        f"Database schema (PK/FK annotated): {schema}\n\n"
        f"Question: {row['question']}\n\n"
        "Return JSON only \u2014 column names without annotations: {\"TableName\": [\"col1\", \"col2\"], ...}"
    )
    answer = json.dumps(row["schema_links"], ensure_ascii=False)
    return {
        "text": (
            f"<|im_start|>system\n{system_prompt}<|im_end|>\n"
            f"<|im_start|>user\n{prompt}<|im_end|>\n"
            f"<|im_start|>assistant\n{answer}<|im_end|>"
        )
    }

In [ ]:
experiment = Experiment(experiment_name="experkrj_conservative_1", mode="fit")

In [ ]:
import torch

QWEN         = "Qwen/Qwen2.5-1.5B-Instruct"
ATTN_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]
ALL_MODULES  = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

configs_spec = [
    ("P1_r4_attn_basic_e2",  QWEN, basic_formatting_function,  4,  8, 1e-5, 2, ATTN_MODULES),
    ("P2_r4_attn_basic_e3",  QWEN, basic_formatting_function,  4,  8, 1e-5, 3, ATTN_MODULES),
    ("P3_r4_all_basic_e2",   QWEN, basic_formatting_function,  4,  8, 1e-5, 2, ALL_MODULES),
    ("P4_r8_attn_basic_e2",  QWEN, basic_formatting_function,  8, 16, 1e-5, 2, ATTN_MODULES),
    ("P5_r8_all_basic_e2",   QWEN, basic_formatting_function,  8, 16, 1e-5, 2, ALL_MODULES),
    ("P6_r4_attn_lr5e6_e2",  QWEN, basic_formatting_function,  4,  8, 5e-6, 2, ATTN_MODULES),
    ("P7_r4_attn_pkfk_e2",   QWEN, pkfk_formatting_function,   4,  8, 1e-5, 2, ATTN_MODULES),
    ("P8_r8_all_pkfk_e3",    QWEN, pkfk_formatting_function,   8, 16, 1e-5, 3, ALL_MODULES),
]

all_configs = []
for label, model_name, fmt_func, r, alpha, lr, num_epochs, target_modules in configs_spec:
    model_kwargs = {
        "torch_dtype": torch.float16,
        "use_cache": False,
    }
    all_configs.append(RFModelConfig(
        model_name=model_name,
        peft_config=RFLoraConfig(
            r=r,
            lora_alpha=alpha,
            lora_dropout=0.1,
            target_modules=target_modules,
            bias="none",
        ),
        training_args=RFSFTConfig(
            learning_rate=lr,
            lr_scheduler_type="linear",
            num_train_epochs=num_epochs,
            per_device_train_batch_size=1,
            per_device_eval_batch_size=1,
            gradient_accumulation_steps=4,
            gradient_checkpointing=False,
            logging_steps=10,
            eval_strategy="steps",
            eval_steps=20,
            bf16=False,
            fp16=True,
        ),
        model_type="causal_lm",
        model_kwargs=model_kwargs,
        formatting_func=fmt_func,
    ))

print(f"Total configs: {len(all_configs)}")
for i, (label, *_) in enumerate(configs_spec):
    print(f"  [{i+1}] {label}")
config_set = List(all_configs)

In [ ]:
def sample_create_model(model_config):
    import gc
    import os
    os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

    from transformers import AutoModelForCausalLM, AutoTokenizer
    import torch

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    model_name = model_config["model_name"]
    model_kwargs = dict(model_config["model_kwargs"])
    model_kwargs.pop("device_map", None)
    model_kwargs.setdefault("low_cpu_mem_usage", True)

    model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    return (model, tokenizer)

In [ ]:
config_group = RFGridSearch(
    configs=config_set,
    trainer_type="SFT"
)

In [ ]:
experiment.run_fit(
    config_group,
    sample_create_model,
    train_dataset,
    validation_dataset,
    num_chunks=1,
    seed=42,
)

In [ ]:
experiment.end()